# Technical Assessment Demo

Partial cross-entropy loss + point-supervised U-Net on aerial imagery.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import torch

ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT))

from src.dataset import AerialSegmentationDataset, sample_point_labels, rgb_mask_to_class_indices
from src.partial_ce_loss import PartialCrossEntropyLoss
from src.model import UNet
from src.constants import CLASS_NAMES, NUM_CLASSES, IGNORE_INDEX
from PIL import Image
import numpy as np

## 1. Partial Cross-Entropy Loss

In [ ]:
pce = PartialCrossEntropyLoss(ignore_index=IGNORE_INDEX)
logits = torch.randn(2, NUM_CLASSES, 32, 32)
targets = torch.randint(0, NUM_CLASSES, (2, 32, 32))
mask = torch.zeros(2, 32, 32)
mask[:, 10:20, 10:20] = 1.0  # only a small region is labeled
loss = pce(logits, targets, mask)
print('PCE loss (sparse region):', loss.item())

## 2. Simulated Point Labels

In [ ]:
img_path = ROOT / 'Data/aerial imagery/Tile 1/images/image_part_001.jpg'
mask_path = ROOT / 'Data/aerial imagery/Tile 1/mask/image_part_001.png'
image = np.array(Image.open(img_path).convert('RGB'))
gt = rgb_mask_to_class_indices(np.array(Image.open(mask_path).convert('RGB')))
points, label_mask = sample_point_labels(gt, num_points=200, strategy='stratified')

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(image)
ax[0].set_title('Image')
ax[1].imshow(gt, cmap='tab10', vmin=0, vmax=5)
ax[1].set_title('Full mask (GT)')
ax[2].imshow(image)
ys, xs = np.where(label_mask > 0)
ax[2].scatter(xs, ys, c=points[ys, xs], cmap='tab10', s=8, vmin=0, vmax=5)
ax[2].set_title('Simulated point labels')
for a in ax: a.axis('off')
plt.tight_layout()
plt.show()

## 3. Train / Evaluate

Run full experiments from terminal:

```bash
python train.py --experiment all --epochs 20
```

In [ ]:
import json
results_path = ROOT / 'outputs/experiment_results.json'
if results_path.exists():
    results = json.loads(results_path.read_text())
    for r in results:
        print(f"{r['name']:30s}  mIoU={r['best_mean_iou']:.4f}")
else:
    print('Run train.py first to generate results.')